**Question 1**
** Remove the duplicates and only keep latest record**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
data=[
    (101, 'Anurag','2026-04-23'),
    (101, 'Anurag','2026-05-03'),
    (102, 'Stella','2026-05-06'),
    (102, 'Stella','2026-06-14'),
    (103, 'Ravi','2026-07-17')
]

In [0]:
columns=['Customer_id', 'Customer_name','Updated_date']

In [0]:
df=spark.createDataFrame(data,columns)
df.display()

Customer_id,Customer_name,Updated_date
101,Anurag,2026-04-23
101,Anurag,2026-05-03
102,Stella,2026-05-06
102,Stella,2026-06-14
103,Ravi,2026-07-17


**Solution for Question 1**


In [0]:
window_spec=Window.partitionBy('Customer_id').orderBy(col('Updated_date').desc())

resultdf=df.withColumn("rn", row_number().over(window_spec)).filter(col("rn")==1).drop("rn")

display(resultdf)

Customer_id,Customer_name,Updated_date
101,Anurag,2026-05-03
102,Stella,2026-06-14
103,Ravi,2026-07-17


follow up que- why not use drop duplicate
Because- we have different different data so only window function can be used.

**Find highest salary Employee per department**

In [0]:
employeee_data=[
    (1,"Anurag","IT",90000),
    (2,"Stella","HR",80000),
    (3,"Ravi","IT",70000),
    (4,"Priya","HR",60000),
    (5,"Rajesh","IT",60000),
    (6,"Priyanka","HR",40000)
]

employee_columns=["emp_id","emp_name","department","salary"]
employee_df=spark.createDataFrame(employeee_data,employee_columns)
employee_df.display()

#window function required
window_spec=Window.partitionBy('department').orderBy(col('salary').desc())

resultdf=employee_df.withColumn("salary_rank", dense_rank().over(window_spec)).filter(col("salary_rank")==1).drop("salary_rank")

display(resultdf)

emp_id,emp_name,department,salary
1,Anurag,IT,90000
2,Stella,HR,80000
3,Ravi,IT,70000
4,Priya,HR,60000
5,Rajesh,IT,60000
6,Priyanka,HR,40000


emp_id,emp_name,department,salary
2,Stella,HR,80000
1,Anurag,IT,90000


**Question 3: Month over Month revenue growth**

In [0]:
sales_data=[
    (101,"2024-01",10000),
    (102,"2024-01",15000),
    (103,"2024-01",20000),
    (101,"2024-02",12000),
    (102,"2024-02",18000),
    (103,"2024-02",22000)
]

sales_columns=["customer_id","month","revenue"]

sales_df=spark.createDataFrame(data=sales_data,schema=sales_columns)
display(sales_df)

window_spec=Window.partitionBy('customer_id').orderBy('month')

result_df=sales_df.withColumn('previours_month_revenue',lag('revenue').over(window_spec)).withColumn('revenue_difference',col('revenue')-col('previours_month_revenue'))

display(result_df)

customer_id,month,revenue
101,2024-01,10000
102,2024-01,15000
103,2024-01,20000
101,2024-02,12000
102,2024-02,18000
103,2024-02,22000


customer_id,month,revenue,previours_month_revenue,revenue_difference
101,2024-01,10000,null,null
101,2024-02,12000,10000,2000
102,2024-01,15000,null,null
102,2024-02,18000,15000,3000
103,2024-01,20000,null,null
103,2024-02,22000,20000,2000


**Question 4: Optimize a Join**


In [0]:
transaction_data=[
    (1,101,"IN",5000),
    (2,102,"US",7000),
    (3,103,"IN",3000),
    (4,101,"UK",9000)
]

country_data=[
    ("IN","India"),
    ("US","United States"),
    ("UK","United Kingdom")
]
transaction_df=spark.createDataFrame(transaction_data,["txn_id","cust_id","country_code","amount"])
country_df=spark.createDataFrame(country_data,["country_code","country_name"])

display(transaction_df)
display(country_df)
#display(transaction_df.join(country_df,'country_code'))
#optimized way to join
display(transaction_df.join(broadcast(country_df),'country_code'))

txn_id,cust_id,country_code,amount
1,101,IN,5000
2,102,US,7000
3,103,IN,3000
4,101,UK,9000


country_code,country_name
IN,India
US,United States
UK,United Kingdom


country_code,txn_id,cust_id,amount,country_name
IN,1,101,5000,India
US,2,102,7000,United States
IN,3,103,3000,India
UK,4,101,9000,United Kingdom


**Question 5: Incremental Loading**

In [0]:
orders_data=[
    (1,"Created","2024-01-01"),
    (2,"Shipped","2024-01-02"),
    (3,"Delivered","2024-01-03"),
    (4,"Cancelled","2024-01-04"),
    (5,"Created","2024-01-05")
]
last_processed_date="2024-01-02"
orders_df=spark.createDataFrame(orders_data,["order_id","status","updated_at"])
display(orders_df)

incremental_df=orders_df.filter(col("updated_at")>last_processed_date)
display(incremental_df)

order_id,status,updated_at
1,Created,2024-01-01
2,Shipped,2024-01-02
3,Delivered,2024-01-03
4,Cancelled,2024-01-04
5,Created,2024-01-05


order_id,status,updated_at
3,Delivered,2024-01-03
4,Cancelled,2024-01-04
5,Created,2024-01-05
